# SOCCAT: Social Group Mention Detection
## mDeBERTa-v3 Fine-Tuning Pipeline

This notebook fine-tunes `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` on a multilingual (French/German) annotated corpus for binary sentence-level social group mention detection.

**To replicate:** update the paths in the *Configuration* cell, then run all cells in order.

**Model weights:** available on the Hugging Face Hub at `selsar/social_group_detection`.

**Reference:** Šarenkapa (2025). *Social Groups Construction in the Media and its Effect on Public Policy.* PhD dissertation, Sciences Po Paris.

## Quick Inference (Skip Training)

If you only want to reproduce the evaluation results, you can load the fine-tuned model directly from the Hugging Face Hub without running the full training pipeline. Jump from this cell straight to **Section 9** (Inference).

> **Requires:** the test JSONL at `TEST_PATH` and the tokenised `tokenized["test"]` dataset — run Sections 3–5 first, then come back here.

In [4]:
HF_MODEL = "selsar/social_group_detection"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(HF_MODEL)

# Re-tokenise with the Hub model's tokenizer
tokenized = dataset.map(
    lambda x: tokenizer(x["text"], padding="max_length", truncation=True),
    batched=True,
)

# Minimal Trainer for inference only
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(OUTPUT_DIR / "tmp"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to=[],
    ),
)

print(f"Loaded '{HF_MODEL}' — continue from Section 9.")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


KeyboardInterrupt: 

## 1. Installation

Run once per environment. `%%capture` suppresses verbose installer output.

In [ ]:
%%capture
!pip install transformers datasets accelerate -U

## 2. Imports

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    auc,
    cohen_kappa_score,
)

## 3. Configuration

Update the paths below to match your environment. Everything else can be left at its default.

| Variable | Description |
|---|---|
| `TRAIN_PATH` | JSONL training file (fields: `text`, `label`) |
| `TEST_PATH` | JSONL test file |
| `OUTPUT_DIR` | Directory for checkpoints, predictions, and result tables |
| `MODEL_NAME` | Base checkpoint on the Hugging Face Hub |
| `SEED` | Global random seed |

In [ ]:
# ── Paths (edit these) ────────────────────────────────────────────────────
TRAIN_PATH = Path("../../data/model_performance/step_1/train_with_all_outlets.json")  # JSONL
TEST_PATH  = Path("../../data/model_performance/step_1/test_with_all_outlets.json")   # JSONL
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 98
set_seed(SEED)

# ── Training hyperparameters ───────────────────────────────────────────────
LEARNING_RATE    = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE  = 64
NUM_EPOCHS       = 4
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06

## 4. Data Loading

Each file is a JSONL where every line is a JSON object with at minimum `text` (str) and `label` (bool or 0/1). Optional metadata fields (`language`, `paper`, `date`, `year`) are preserved for sliced evaluation in Section 10.

In [ ]:
def read_json_lines(filepath):
    """Read a JSONL file into a list of dicts."""
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def process_data(data):
    """Return list of {text, label} dicts with integer labels."""
    return [{"text": item["text"], "label": int(bool(item["label"]))} for item in data]


raw_train = read_json_lines(TRAIN_PATH)
raw_test  = read_json_lines(TEST_PATH)

train_dataset = Dataset.from_pandas(pd.DataFrame(process_data(raw_train)))
test_dataset  = Dataset.from_pandas(pd.DataFrame(process_data(raw_test)))

dataset = DatasetDict({"train": train_dataset, "test": test_dataset})
print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")

## 5. Tokenisation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_function, batched=True)
print("Tokenisation complete.")

## 6. Model & Evaluation Metrics

The NLI classification head is replaced with a two-class linear layer. `compute_metrics_trainer` is called by the HF `Trainer` after each evaluation epoch.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
model.classifier = torch.nn.Linear(model.config.hidden_size, 2)
model.num_labels = 2


def compute_metrics_trainer(p):
    """Metrics callback for HF Trainer (input is EvalPrediction)."""
    preds, labels = p.predictions, p.label_ids
    pred_labels = preds.argmax(axis=1)

    # PR-AUC (micro-averaged over both classes)
    try:
        n_classes = preds.shape[1]
        y_true_bin = np.eye(n_classes)[labels]
        prec_curve, rec_curve, _ = precision_recall_curve(y_true_bin.ravel(), preds.ravel())
        pr_auc = auc(rec_curve, prec_curve)
    except Exception:
        pr_auc = float("nan")

    return {
        "accuracy":             accuracy_score(labels, pred_labels),
        "precision_weighted":   precision_score(labels, pred_labels, average="weighted", zero_division=0),
        "recall_weighted":      recall_score(labels, pred_labels, average="weighted", zero_division=0),
        "f1_weighted":          f1_score(labels, pred_labels, average="weighted", zero_division=0),
        "f1_macro":             f1_score(labels, pred_labels, average="macro", zero_division=0),
        "f1_micro":             f1_score(labels, pred_labels, average="micro", zero_division=0),
        "cohen_kappa":          cohen_kappa_score(labels, pred_labels),
        "pr_auc":               pr_auc,
    }

## 7. Training

Fine-tunes on the training split; evaluates on the test split at the end of every epoch. Checkpoints and logs are written to `OUTPUT_DIR/`.

To enable Weights & Biases logging, change `report_to=[]` to `report_to=["wandb"]` and run `wandb.login()` before this cell.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    logging_dir=str(OUTPUT_DIR / "logs"),
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics_trainer,
)

trainer.train()

## 8. Save Fine-Tuned Model

In [ ]:
model_save_path = OUTPUT_DIR / "mDeBERTa_social_group_detection"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to: {model_save_path}")

## 9. Inference & Metadata Assembly

Runs prediction on the test set and attaches metadata (outlet, country, decade) from the raw JSONL for sliced evaluation.

In [ ]:
# ── Metadata helpers ──────────────────────────────────────────────────────
def extract_year(row):
    """Extract a 4-digit year from an explicit year field or a date string."""
    y = row.get("year")
    if pd.notna(y):
        try:
            return int(float(y))
        except (TypeError, ValueError):
            pass
    d = row.get("date")
    if d is None or pd.isna(d):
        return None
    m = re.search(r"(19|20)\d{2}", str(d))
    return int(m.group(0)) if m else None


def year_to_decade(y):
    if y is None or pd.isna(y):
        return "Unknown"
    return f"{(int(y) // 10) * 10}s"


def map_country(lang):
    return {"French": "France", "German": "Germany"}.get(lang, "Unknown")


# ── Run predictions ───────────────────────────────────────────────────────
pred_output = trainer.predict(tokenized["test"])
preds = np.argmax(pred_output.predictions, axis=1)

# ── Assemble results DataFrame ────────────────────────────────────────────
results_df = pd.DataFrame(raw_test)
results_df["pred"]      = preds
results_df["true"]      = test_dataset["label"]
results_df["language"]  = results_df.get("language", pd.Series(dtype=str)).fillna("Unknown")
results_df["country"]   = results_df["language"].map(map_country)
results_df["paper"]     = results_df.get("paper", pd.Series(dtype=str)).fillna("Unknown")
results_df["year_clean"] = results_df.apply(extract_year, axis=1)
results_df["decade"]    = results_df["year_clean"].apply(year_to_decade)

print(f"Predictions complete. Test sentences: {len(results_df):,}")

## 10. Sliced Evaluation

Computes weighted precision, recall, F1, macro F1, and accuracy overall and by outlet, country, and decade.

In [ ]:
def metrics_for_group(df):
    y_true = df["true"].astype(int)
    y_pred = df["pred"].astype(int)
    return {
        "Accuracy":             accuracy_score(y_true, y_pred),
        "Precision_weighted":   precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_weighted":      recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_weighted":          f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_macro":             f1_score(y_true, y_pred, average="macro", zero_division=0),
        "N":                    len(df),
    }


def compute_sliced_metrics(df, group_cols):
    rows = []
    for keys, grp in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        rec = metrics_for_group(grp)
        for col, val in zip(group_cols, keys):
            rec[col] = val
        rows.append(rec)
    return pd.DataFrame(rows)


overall_df        = pd.DataFrame([metrics_for_group(results_df)])
outlet_df         = compute_sliced_metrics(results_df, ["paper"]).sort_values("F1_weighted", ascending=False)
country_df        = compute_sliced_metrics(results_df, ["country"]).sort_values("F1_weighted", ascending=False)
decade_df         = compute_sliced_metrics(results_df, ["decade"]).sort_values("decade")
outlet_decade_df  = compute_sliced_metrics(results_df, ["paper", "decade"])

print("=== Overall ===")
print(overall_df.to_string(index=False))
print("\n=== By Outlet ===")
print(outlet_df.to_string(index=False))

## 11. Save Results

In [ ]:
results_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

overall_df.to_csv(OUTPUT_DIR / "performance_overall.csv", index=False)
outlet_df.to_csv(OUTPUT_DIR / "performance_per_outlet.csv", index=False)
country_df.to_csv(OUTPUT_DIR / "performance_per_country.csv", index=False)
decade_df.to_csv(OUTPUT_DIR / "performance_per_decade.csv", index=False)
outlet_decade_df.to_csv(OUTPUT_DIR / "performance_outlet_x_decade.csv", index=False)

excel_path = OUTPUT_DIR / "model_performance_summary.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    overall_df.to_excel(writer, sheet_name="Overall", index=False)
    outlet_df.to_excel(writer, sheet_name="Per_Outlet", index=False)
    country_df.to_excel(writer, sheet_name="Per_Country", index=False)
    decade_df.to_excel(writer, sheet_name="Per_Decade", index=False)
    outlet_decade_df.to_excel(writer, sheet_name="Outlet_x_Decade", index=False)

print(f"All results saved to: {OUTPUT_DIR}")

## 12. (Optional) Push to Hugging Face Hub

Uncomment and run this cell to upload the fine-tuned model. `notebook_login()` will prompt for your HF access token.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# HF_REPO_NAME = "your-username/social_group_detection"  # <-- set this
# model.push_to_hub(HF_REPO_NAME)
# tokenizer.push_to_hub(HF_REPO_NAME)

## 13. Visualisation

Bar charts of weighted F1 and accuracy by outlet, and metric comparison by country. Figures are also saved as PDFs in `OUTPUT_DIR`.

In [ ]:
sns.set_theme(style="whitegrid")

# ── F1 and accuracy by outlet ─────────────────────────────────────────────
plot_df = outlet_df.merge(
    results_df[["paper", "country"]].drop_duplicates(), on="paper", how="left"
)

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(plot_df) * 0.45)))
for ax, metric, title in zip(
    axes,
    ["F1_weighted", "Accuracy"],
    ["Weighted F1 per Outlet", "Accuracy per Outlet"],
):
    sns.barplot(data=plot_df, x=metric, y="paper", hue="country", dodge=False, ax=ax)
    ax.set_xlim(0.7, 1.0)
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.set_ylabel("Outlet")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "performance_by_outlet.pdf", bbox_inches="tight")
plt.show()

# ── Metrics by country ────────────────────────────────────────────────────
country_melted = country_df.melt(
    id_vars="country",
    value_vars=["Accuracy", "F1_weighted", "F1_macro"],
    var_name="Metric",
    value_name="Score",
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=country_melted, x="Metric", y="Score", hue="country", ax=ax)
ax.set_ylim(0.7, 1.0)
ax.set_title("Performance by Country")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "performance_by_country.pdf", bbox_inches="tight")
plt.show()